# 실습 — 합성 결함 데이터 만들기 (Diffusion 생성 / 파인튜닝)

이 노트북은 수업 심화 ①·②에 **정확히 대응**하는 실습입니다. 모두 Hugging Face `diffusers`로 **무료 Colab T4**에서 동작합니다.

| 파트 | 대응 주제 | 무엇을 하나 | 학습 필요? | 무게 |
| --- | --- | --- | --- | --- |
| **Part A** | ② 텍스트 기반 생성 | 정상 이미지에 **"텍스트(무엇) + 마스크(어디)"** 로 결함을 그려 넣기 (inpainting) | ❌ 불필요 | 가벼움 (몇 분) |
| **Part B** | ① 파인튜닝 | 소수 결함 이미지로 모델에 **우리 결함을 가르치기** (LoRA DreamBooth) | ✅ 필요 | 무거움 (수십 분) |

**핵심 메시지** — 두 파트 모두 "그럴듯한 그림"이 아니라 **결함 이미지 + 위치 마스크(=라벨)** 를 만들어 탐지기 학습에 쓰는 것이 목적입니다.

> **시작 전**
>
> 상단 메뉴 **런타임 → 런타임 유형 변경 → GPU(T4)** 로 설정하세요.


## 0. 환경 설정

In [ ]:
# 필요한 라이브러리 설치
!pip install -q diffusers transformers accelerate peft safetensors

import torch
print("GPU 사용 가능:", torch.cuda.is_available())
!nvidia-smi -L  # 어떤 GPU가 붙었는지 확인 (보통 Tesla T4)

---
# Part A — ② 텍스트 + 마스크로 결함 생성 (Inpainting)

**아이디어**: 정상 제품 이미지를 두고, "여기"(**마스크**)에 "이런 결함"(**텍스트**)을 그려 넣습니다.
- 텍스트 = 결함의 *종류/외형* ("긁힌 자국", "찍힘", "오염")
- 마스크 = 결함의 *위치* (흰색 = 결함을 넣을 영역, 검은색 = 그대로 유지)
- **이 마스크가 곧 정답 라벨**이 됩니다 → 이미지+마스크 쌍 확보!

> **참고**
>
> 이것이 인트로 타임라인의 "텍스트＋마스크"(Phase 3) 단계를 가장 단순하게 체험하는 방법입니다.

### A-1. 정상(결함 없는) 베이스 이미지 준비

In [ ]:
# 실습 편의를 위해 텍스트→이미지로 "정상 표면"을 생성합니다.
# (실무에서는 이 부분을 본인 제품의 정상 이미지 업로드로 바꾸세요.)
from diffusers import AutoPipelineForText2Image

t2i = AutoPipelineForText2Image.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
).to("cuda")

base = t2i(
    "a clean brushed metal plate, top-down view, even lighting, no defects, industrial",
    height=512, width=512, num_inference_steps=30, guidance_scale=7.5,
).images[0]
base.save("/content/normal.png")
base

# --- (대안) 본인 정상 이미지를 쓰려면 아래 주석을 해제 ---
# from google.colab import files
# from PIL import Image
# up = files.upload()
# base = Image.open(next(iter(up))).convert("RGB").resize((512, 512))
# base.save("/content/normal.png")

### A-2. 마스크 만들기 (= 결함을 넣을 위치 = 라벨)

In [ ]:
from PIL import Image, ImageDraw

# 검은 배경(유지) 위에 흰색 영역(결함을 그릴 곳)을 그립니다.
mask = Image.new("L", (512, 512), 0)          # 0 = 검정 = 원본 유지
draw = ImageDraw.Draw(mask)
draw.ellipse([200, 180, 330, 250], fill=255)  # 255 = 흰색 = 결함 생성 영역
mask.save("/content/mask.png")
mask

# 위치를 바꾸려면 [x1, y1, x2, y2] 좌표를 수정하세요.
# 사각형 결함을 원하면 draw.rectangle([...], fill=255) 로 변경.

### A-3. 텍스트 + 마스크로 결함 생성

In [ ]:
from diffusers import AutoPipelineForInpainting

inpaint = AutoPipelineForInpainting.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-inpainting",
    torch_dtype=torch.float16,
).to("cuda")

# 텍스트 = "무엇을" 그릴지. 결함 종류를 바꿔가며 실험해 보세요.
prompt = "a deep scratch and a dent, damaged metal surface, defect, scratched"

result = inpaint(
    prompt=prompt,
    image=base,           # 정상 베이스 이미지
    mask_image=mask,      # 흰색 영역에만 결함 생성
    guidance_scale=8.0,   # 클수록 프롬프트를 강하게 따름
    num_inference_steps=40,
).images[0]
result.save("/content/defect.png")
result

### A-4. 결과 비교 — "결함이 어디에 추가됐나"를 눈으로 확인

단순히 정상/결함을 나란히 보는 데서 그치지 않고, **마스크 영역을 중심으로** 비교합니다.
이렇게 하면 *"정확히 어느 부분이 바뀌었는지"* 를 직접 눈으로 확인할 수 있습니다.

- 위쪽: 정상·생성 이미지에 **마스크 영역을 빨갛게 표시** + 바뀐 픽셀을 보여주는 **차이맵**
- 아래쪽: **마스크 영역만 확대**한 정상 vs 생성 (결함이 들어간 자리)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# PIL → numpy
base_np = np.array(base.convert("RGB"))
res_np  = np.array(result.convert("RGB"))
mask_np = np.array(mask.convert("L"))
m = mask_np > 127                      # 결함을 넣은 영역(True)

# 1) 마스크 영역을 빨갛게 오버레이한 이미지 만들기
def overlay_mask(img, m, alpha=0.40):
    out = img.copy().astype(np.float32)
    out[m] = (1 - alpha) * out[m] + alpha * np.array([255, 0, 0])
    return out.astype(np.uint8)

base_ov = overlay_mask(base_np, m)
res_ov  = overlay_mask(res_np, m)

# 2) 바뀐 픽셀(차이맵): 생성-정상의 절대차 합 → 마스크 영역에 집중되는지 확인
diff = np.abs(res_np.astype(int) - base_np.astype(int)).sum(axis=2)

# 3) 마스크의 경계 상자(bbox)로 영역만 확대
ys, xs = np.where(m)
pad = 20
y0, y1 = max(ys.min() - pad, 0), min(ys.max() + pad, base_np.shape[0])
x0, x1 = max(xs.min() - pad, 0), min(xs.max() + pad, base_np.shape[1])
base_crop = base_np[y0:y1, x0:x1]
res_crop  = res_np[y0:y1, x0:x1]

# --- 시각화 (2행 3열) ---
fig, ax = plt.subplots(2, 3, figsize=(13, 9))
ax[0, 0].imshow(base_ov);  ax[0, 0].set_title("Normal + mask region (red)")
ax[0, 1].imshow(res_ov);   ax[0, 1].set_title("Generated + mask region (red)")
im = ax[0, 2].imshow(diff, cmap="magma")
ax[0, 2].set_title("Difference map (brighter = more change)")
fig.colorbar(im, ax=ax[0, 2], fraction=0.046)

ax[1, 0].imshow(base_crop); ax[1, 0].set_title("[Zoom] Normal - where defect goes")
ax[1, 1].imshow(res_crop);  ax[1, 1].set_title("[Zoom] Generated - added defect")
ax[1, 2].imshow(mask, cmap="gray"); ax[1, 2].set_title("Mask = label")

for a in ax.ravel():
    a.axis("off")
plt.tight_layout(); plt.show()

# 차이맵이 마스크 안에 얼마나 집중됐는지 수치로도 확인
inside = diff[m].mean()
outside = diff[~m].mean() + 1e-6
print(f"마스크 안 평균 변화: {inside:.1f} / 바깥 평균 변화: {outside:.1f} "
      f"→ 안쪽이 약 {inside/outside:.1f}배 더 크게 바뀜 (결함이 마스크 영역에 들어갔다는 뜻)")

**Part A 정리**
- **텍스트**가 결함의 *종류*를, **마스크**가 *위치*를 결정했습니다.
- 마스크는 그대로 **세그멘테이션 정답 라벨**이 됩니다 → 사람이 픽셀을 칠하는 비싼 라벨링을 줄여 줍니다.
- 프롬프트와 마스크 위치/모양만 바꾸면 **다양한 결함을 무한히** 찍어낼 수 있습니다.
- 한계: 범용 모델이라 *우리 제품 특유의 결함*은 어색할 수 있습니다 → 그래서 Part B(파인튜닝)가 필요합니다.

---
# Part B — ① 파인튜닝(LoRA)으로 "우리 결함" 가르치기

**아이디어**: 소수의 실제 결함 이미지로 모델을 살짝 적응(LoRA)시켜, *우리 도메인 결함*을 충실히 생성하게 만듭니다.

> **무료 Colab 주의**
>
> 학습이 들어가므로 무겁습니다. 아래는 빠른 **데모 설정(400 steps)** 입니다. 실제 품질을 내려면 더 많은 step이 필요하고, 그만큼 시간이 듭니다. 세션이 끊기지 않게 탭을 열어 두세요.

### B-1. 학습 데이터 준비

파인튜닝에는 **같은 종류의 결함 이미지 5~10장**이 필요합니다. 아래 두 방법 중 **하나**를 쓰세요.

**예시로 쓸 표준 데이터셋 두 가지** (둘 다 산업 이상탐지 벤치마크, 라이선스 CC BY-NC-SA 4.0 — 연구·비상업용):

- **MVTec AD** — MVTec 공개. 15개 카테고리(텍스처 5 + 객체 10), 총 5,354장. 학습셋은 정상 이미지만, 테스트셋에 정상＋결함이 섞이고 70여 종 결함에 **픽셀 단위 마스크**가 달려 있습니다. 이상탐지의 *사실상 표준 벤치마크*입니다.
- **VisA** — Amazon이 2022년 공개. 12개 카테고리(candle, capsules, cashew, chewinggum, fryum, macaroni1/2, pcb1~4, pipe_fryum), 총 10,821장(정상 9,621 + 이상 1,200), **이미지·픽셀 라벨** 제공. 복수 인스턴스·복잡한 배경이 많아 MVTec보다 어렵습니다.

아래 **Option A**는 Part A에서 합성한 **금속 표면 결함**과 결을 맞춰, MVTec의 금속 부품 카테고리인 **`metal_nut`** 를 사용합니다. (VisA에는 베어링·너트 같은 금속 부품이 없어 metal_nut가 가장 가깝습니다.) 전체 MVTec(4.9GB)을 받지 않고, **metal_nut 서브셋(약 166MB)** 만 올라온 Hugging Face 데이터셋에서 결함 이미지 몇 장만 받습니다 — API 키가 필요 없습니다.

In [ ]:
# === Option A: MVTec metal_nut 일부만 받기 (Hugging Face, 키 불필요) ===
# 전체 MVTec(4.9GB) 대신 metal_nut 서브셋(약 166MB)만 받고, 그중 '결함' 이미지 몇 장만 사용합니다.
!pip install -q datasets
import os, random
from datasets import load_dataset

DEFECT   = "scratch"   # Part A가 스크래치를 합성하므로 동일 유형으로 맞춤 (bent/color/flip/scratch 중 택1, None이면 전체 결함)
N_IMAGES = 12
INSTANCE_DIR = "/content/defect_imgs"
os.makedirs(INSTANCE_DIR, exist_ok=True)

# metal_nut의 결함 이미지는 test split에 있습니다 (train은 정상만).
ds = load_dataset("MSherbinii/mvtec-ad-metal-nut", split="test")
names  = ds.features["label"].names         # 예: ['bent','color','flip','good','scratch']
labels = ds["label"]                         # 이미지 로드 없이 라벨만
good   = names.index("good") if "good" in names else -1
print("결함 유형:", [n for n in names if n != "good"])

# 원하는 결함 유형 우선 선택, 부족하면 다른 결함으로 보충
if DEFECT in names:
    sel = [i for i, l in enumerate(labels) if l == names.index(DEFECT)]
else:
    sel = [i for i, l in enumerate(labels) if l != good]
random.seed(0); random.shuffle(sel)
if len(sel) < N_IMAGES:
    extra = [i for i, l in enumerate(labels) if l != good and i not in sel]
    random.shuffle(extra); sel += extra[:N_IMAGES - len(sel)]

# 선택한 N장만 저장
for k, i in enumerate(sel[:N_IMAGES]):
    img = ds[i]["image"].convert("RGB")
    img.save(os.path.join(INSTANCE_DIR, f"metal_nut_{names[labels[i]]}_{k:02d}.jpg"))
print("준비된 학습 이미지:", len(os.listdir(INSTANCE_DIR)))

In [ ]:
# === Option B: 본인 결함 이미지 직접 업로드 (Option A 대신) ===
import os
from google.colab import files

INSTANCE_DIR = "/content/defect_imgs"
os.makedirs(INSTANCE_DIR, exist_ok=True)

# 같은 종류의 결함 이미지 5~10장을 업로드하세요.
uploaded = files.upload()
for fn in list(uploaded.keys()):
    os.replace(fn, os.path.join(INSTANCE_DIR, fn))
print("업로드된 이미지 수:", len(os.listdir(INSTANCE_DIR)))

### B-2. 공식 LoRA DreamBooth 학습 스크립트 준비

In [ ]:
# Hugging Face diffusers 공식 예제 스크립트를 받아옵니다.
!git clone -q https://github.com/huggingface/diffusers /content/diffusers
%cd /content/diffusers/examples/dreambooth
!pip install -q -r requirements.txt

### B-3. LoRA 파인튜닝 실행 (데모 설정)

> **예상 시간**
>
> 무료 Colab T4 기준, **첫 실행은 약 8~20분**(SD1.5 가중치 ~4GB 다운로드 포함), 이후 재실행은 학습만 **약 3~8분**입니다(400 steps 기준). **30초~1분 만에 끝났다면 학습이 실제로 돌지 않고 오류로 종료된 것**입니다 → 셀 출력 **위쪽의 에러 메시지**를 확인하세요. 아래 셀은 그런 경우 마지막에 명확히 알려주도록 자가 점검을 넣었습니다.

In [ ]:
import os, glob

INSTANCE_DIR = "/content/defect_imgs"
OUTPUT_DIR   = "/content/defect_lora"

# (1) 학습 전 점검 — 이미지가 비어 있으면 학습이 곧바로 실패합니다 (B-1을 먼저 실행했는지 확인)
imgs = [f for f in os.listdir(INSTANCE_DIR)] if os.path.isdir(INSTANCE_DIR) else []
assert len(imgs) > 0, f"학습 이미지가 없습니다 ({INSTANCE_DIR}). 먼저 B-1(Option A 또는 B)을 실행하세요."
print(f"학습 이미지 {len(imgs)}장으로 시작 — T4 기준 첫 실행 약 8~20분(모델 다운로드 포함) 예상\n")

In [ ]:
# (2) 학습 실행 — "sks"는 모델이 모르는 희귀 토큰 → "우리 결함"을 가리키는 새 이름표
# 프롬프트는 데이터에 맞게 바꾸세요. (지금은 metal_nut scratch에 맞춰 둠)
!accelerate launch train_dreambooth_lora.py \
  --pretrained_model_name_or_path="stable-diffusion-v1-5/stable-diffusion-v1-5" \
  --instance_data_dir="/content/defect_imgs" \
  --output_dir="/content/defect_lora" \
  --instance_prompt="a photo of sks metal nut with a scratch defect" \
  --resolution=512 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=1 \
  --gradient_checkpointing \
  --mixed_precision="fp16" \
  --learning_rate=1e-4 \
  --lr_scheduler="constant" \
  --lr_warmup_steps=0 \
  --max_train_steps=400 \
  --checkpointing_steps=200 \
  --seed=0

In [ ]:
# (3) 학습 후 점검 — 가중치가 안 생겼으면 위 로그에 에러가 있는 것입니다
import glob, os
weights = glob.glob(os.path.join("/content/defect_lora", "*.safetensors"))
assert weights, (
    "LoRA 가중치가 생성되지 않았습니다 → 학습이 도중에 실패했습니다.\n"
    "  • 위 (2) 셀 출력의 에러 메시지를 확인하세요.\n"
    "  • 30~40초처럼 너무 빨리 끝났다면 학습이 아예 안 돈 것입니다.\n"
    "  • 흔한 원인: B-1 미실행(이미지 0장) / B-2 의존성 설치 실패 / accelerate 인자 오류")
print("LoRA 가중치 저장됨:", weights)
# 품질이 부족하면 --max_train_steps 를 800~1500 으로 올려보세요 (시간 증가).

### B-4. 학습한 LoRA로 우리 결함 생성

In [ ]:
import os, glob, torch
from diffusers import AutoPipelineForText2Image

OUTPUT_DIR = "/content/defect_lora"
weights = glob.glob(os.path.join(OUTPUT_DIR, "*.safetensors"))
assert weights, f"{OUTPUT_DIR}에 LoRA 가중치가 없습니다. B-3을 먼저 성공적으로 마치세요."

pipe = AutoPipelineForText2Image.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
).to("cuda")

# 로컬 폴더의 LoRA 가중치를 '파일명까지 명시'해서 로드
# (파일명을 안 주면 경로를 HF 저장소 id로 오인해 HFValidationError가 납니다)
pipe.load_lora_weights(OUTPUT_DIR, weight_name=os.path.basename(weights[0]))

img = pipe(
    "a photo of sks metal nut with a scratch defect",
    num_inference_steps=30, guidance_scale=7.5,
).images[0]
img.save("/content/defect_finetuned.png")
img

**Part B 정리**
- LoRA는 본체 가중치를 얼리고 **작은 보조 모듈만** 학습해, 무료 T4에서도 파인튜닝이 가능합니다.
- 새 토큰(`sks`)으로 "우리 결함"을 모델에 이름표 붙여 가르쳤습니다.
- 더 가벼운 대안: **Textual Inversion**(`textual_inversion.py`) — 단어 임베딩 하나만 학습.

---
## 마무리 — 두 방법의 매핑과 다음 단계

| | Part A (Inpainting) | Part B (LoRA 파인튜닝) |
| --- | --- | --- |
| 대응 주제 | ② 텍스트 기반 생성 | ① 파인튜닝 |
| 한 줄 | 말 + 위치로 **즉석에서** 결함 생성 | 모델에 **우리 결함을 학습** |
| 학습 | 불필요 (빠름) | 필요 (무거움) |
| 산출물 | 이미지 + 마스크(라벨) | 우리 도메인에 충실한 결함 이미지 |

**실무에서는 둘을 결합**합니다 — 파인튜닝으로 우리 결함을 익힌 모델에 "텍스트＋마스크"로 지시해, **마스크가 정렬된 이미지-라벨 쌍**을 대량 생산합니다.

### 연구급 참고 (둘을 한 번에 하는 결함 특화 코드)
- **AnomalyDiffusion** (AAAI 2024) — 외형(텍스트 임베딩)과 위치(마스크)를 분리 학습해 **이미지-마스크 쌍**을 few-shot으로 생성. 코드: `github.com/sjtuplayer/anomalydiffusion`
- 자동 마스크가 필요하면 **SAM**(Segment Anything)이나 **CLIPSeg**로 마스크를 만들어 Part A에 연결할 수 있습니다.

### "잘" 생성했는지 항상 확인
생성물이 (1) 진짜 같고 (2) 다양하고 (3) 마스크와 정확히 맞는지 — 그리고 최종적으로 **이 합성 데이터로 학습한 탐지기 성능이 실제로 오르는지**가 유일한 시험입니다.